# Category Family Embeddings

This notebook creates embeddings for category families extracted from category codes.
Category families are the first part of the category_code (e.g., 'electronics' from 'electronics.clock').

The notebook demonstrates:
1. Data preparation and category family extraction
2. Different embedding techniques (One-Hot, Label Encoding, Word2Vec, etc.)
3. Embedding evaluation and visualization
4. Saving embeddings for downstream use

In [1]:
# Core libraries
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# For word embeddings
from gensim.models import Word2Vec, FastText
from gensim.models.callbacks import CallbackAny2Vec

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Plotting settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

ModuleNotFoundError: No module named 'gensim'

## Data Loading and Preparation

In [ ]:
def extract_category_family(category_code: str) -> str:
    """
    Extract the category family from a category code.
    The category family is the first part before the dot.
    
    Args:
        category_code: String in format 'family.subcategory' or just 'family'
        
    Returns:
        The category family (first part before the dot)
    """
    if pd.isna(category_code) or category_code == '':
        return 'unknown'
    
    # Split by dot and take the first part
    parts = str(category_code).split('.')
    return parts[0].lower() if parts[0] else 'unknown'

# Create sample dataset with more realistic category codes
def create_sample_dataset(n_samples: int = 10000) -> pd.DataFrame:
    """
    Create a sample dataset with category codes and product features.
    """
    # Define category families and their subcategories
    category_structure = {
        'electronics': ['phone', 'computer', 'tv', 'audio', 'camera', 'gaming', 'wearable'],
        'furniture': ['chair', 'table', 'sofa', 'bed', 'cabinet', 'shelf', 'desk'],
        'clothing': ['shirt', 'pants', 'dress', 'jacket', 'shoes', 'accessories', 'underwear'],
        'home': ['kitchen', 'bathroom', 'decor', 'lighting', 'cleaning', 'garden', 'storage'],
        'sports': ['fitness', 'outdoor', 'team', 'water', 'winter', 'cycling', 'running'],
        'books': ['fiction', 'nonfiction', 'educational', 'children', 'comics', 'academic', 'reference'],
        'toys': ['educational', 'action', 'dolls', 'games', 'puzzles', 'outdoor', 'infant'],
        'beauty': ['skincare', 'makeup', 'hair', 'fragrance', 'tools', 'body', 'men']
    }
    
    # Generate category codes
    category_codes = []
    for family, subcategories in category_structure.items():
        for subcat in subcategories:
            # Add some variations
            category_codes.append(f'{family}.{subcat}')
            category_codes.append(f'{family}.{subcat}.premium')
            category_codes.append(f'{family}.{subcat}.basic')
    
    # Generate random data
    data = {
        'product_id': range(1, n_samples + 1),
        'category_code': np.random.choice(category_codes, n_samples),
        'price': np.random.lognormal(4, 1, n_samples),
        'rating': np.random.beta(5, 2, n_samples) * 5,
        'sales_count': np.random.poisson(100, n_samples),
        'customer_reviews': np.random.poisson(50, n_samples)
    }
    
    df = pd.DataFrame(data)
    
    # Extract category families
    df['category_family'] = df['category_code'].apply(extract_category_family)
    
    return df

# Load or create dataset
print("Creating sample dataset...")
df = create_sample_dataset(10000)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Analyze category families
print("Category Family Analysis:")
print("=" * 50)

family_stats = df.groupby('category_family').agg({
    'product_id': 'count',
    'price': ['mean', 'std'],
    'rating': 'mean',
    'sales_count': 'sum'
}).round(2)

family_stats.columns = ['count', 'avg_price', 'std_price', 'avg_rating', 'total_sales']
family_stats = family_stats.sort_values('count', ascending=False)

print(f"\nNumber of unique category families: {df['category_family'].nunique()}")
print(f"\nCategory family statistics:")
family_stats

## Embedding Techniques

We'll explore different embedding techniques for category families:

### 1. Label Encoding

In [ ]:
def create_label_encoding(df: pd.DataFrame, column: str = 'category_family') -> Dict[str, np.ndarray]:
    """
    Create label encoding for category families.
    """
    le = LabelEncoder()
    labels = le.fit_transform(df[column])
    
    # Create embedding dictionary
    embedding_dict = {}
    for i, family in enumerate(le.classes_):
        embedding_dict[family] = np.array([i])
    
    return {
        'embeddings': embedding_dict,
        'encoder': le,
        'labels': labels,
        'dimension': 1
    }

label_results = create_label_encoding(df)
print(f"Label encoding created with {len(label_results['embeddings'])} families")
print(f"Sample embeddings:")
for family, embedding in list(label_results['embeddings'].items())[:5]:
    print(f"{family}: {embedding}")

### 2. One-Hot Encoding

In [ ]:
def create_onehot_encoding(df: pd.DataFrame, column: str = 'category_family') -> Dict[str, np.ndarray]:
    """
    Create one-hot encoding for category families.
    """
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    onehot = ohe.fit_transform(df[[column]])
    
    # Create embedding dictionary
    embedding_dict = {}
    for i, family in enumerate(ohe.categories_[0]):
        embedding_dict[family] = onehot[df[column] == family][0]
    
    return {
        'embeddings': embedding_dict,
        'encoder': ohe,
        'matrix': onehot,
        'dimension': len(ohe.categories_[0])
    }

onehot_results = create_onehot_encoding(df)
print(f"One-hot encoding created with dimension {onehot_results['dimension']}")
print(f"Sample embeddings:")
for family, embedding in list(onehot_results['embeddings'].items())[:3]:
    print(f"{family}: {embedding[:10]}... (showing first 10 of {len(embedding)} dimensions)")

### 3. Feature-Based Embeddings

In [ ]:
def create_feature_based_embeddings(df: pd.DataFrame, 
                                   column: str = 'category_family',
                                   feature_columns: List[str] = None) -> Dict[str, np.ndarray]:
    """
    Create embeddings based on aggregated features for each category family.
    """
    if feature_columns is None:
        feature_columns = ['price', 'rating', 'sales_count', 'customer_reviews']
    
    # Aggregate features by category family
    agg_features = df.groupby(column)[feature_columns].agg([
        'mean', 'std', 'min', 'max', 'median'
    ]).fillna(0)
    
    # Flatten column names
    agg_features.columns = [f'{col}_{stat}' for col, stat in agg_features.columns]
    
    # Normalize features
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    normalized_features = scaler.fit_transform(agg_features)
    
    # Create embedding dictionary
    embedding_dict = {}
    for i, family in enumerate(agg_features.index):
        embedding_dict[family] = normalized_features[i]
    
    return {
        'embeddings': embedding_dict,
        'features': agg_features,
        'scaler': scaler,
        'dimension': normalized_features.shape[1]
    }

feature_results = create_feature_based_embeddings(df)
print(f"Feature-based embeddings created with dimension {feature_results['dimension']}")
print(f"Sample embeddings:")
for family, embedding in list(feature_results['embeddings'].items())[:3]:
    print(f"{family}: {embedding[:5]}... (showing first 5 of {len(embedding)} dimensions)")

### 4. Word2Vec Embeddings

In [ ]:
def prepare_sequences_for_word2vec(df: pd.DataFrame, 
                                  column: str = 'category_code') -> List[List[str]]:
    """
    Prepare sequences for Word2Vec training.
    """
    sequences = []
    
    # Create sequences based on category codes and their context
    for _, row in df.iterrows():
        category_code = row[column]
        family = extract_category_family(category_code)
        
        # Split category code into components
        components = str(category_code).split('.')
        
        # Create different types of sequences
        # 1. Hierarchical sequence
        sequences.append(components)
        
        # 2. Family-based sequence (products with similar families)
        similar_products = df[df['category_family'] == family]['category_code'].head(5).tolist()
        sequences.append([str(code).split('.')[0] for code in similar_products])
    
    return sequences

def create_word2vec_embeddings(df: pd.DataFrame, 
                                vector_size: int = 50,
                                window: int = 3,
                                min_count: int = 1) -> Dict[str, np.ndarray]:
    """
    Create Word2Vec embeddings for category families.
    """
    # Prepare sequences
    sequences = prepare_sequences_for_word2vec(df)
    
    # Train Word2Vec model
    model = Word2Vec(
        sentences=sequences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=1,  # Skip-gram
        epochs=100,
        seed=RANDOM_SEED
    )
    
    # Create embedding dictionary for category families
    embedding_dict = {}
    for family in df['category_family'].unique():
        if family in model.wv:
            embedding_dict[family] = model.wv[family]
    
    return {
        'embeddings': embedding_dict,
        'model': model,
        'dimension': vector_size
    }

word2vec_results = create_word2vec_embeddings(df)
print(f"Word2Vec embeddings created with dimension {word2vec_results['dimension']}")
print(f"Number of families with embeddings: {len(word2vec_results['embeddings'])}")
print(f"Sample embeddings:")
for family, embedding in list(word2vec_results['embeddings'].items())[:3]:
    print(f"{family}: {embedding[:5]}... (showing first 5 of {len(embedding)} dimensions)")

### 5. FastText Embeddings

In [ ]:
def create_fasttext_embeddings(df: pd.DataFrame, 
                                vector_size: int = 50,
                                window: int = 3,
                                min_count: int = 1) -> Dict[str, np.ndarray]:
    """
    Create FastText embeddings for category families.
    FastText can handle out-of-vocabulary words better than Word2Vec.
    """
    # Prepare sequences
    sequences = prepare_sequences_for_word2vec(df)
    
    # Train FastText model
    model = FastText(
        sentences=sequences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=1,  # Skip-gram
        epochs=100,
        seed=RANDOM_SEED
    )
    
    # Create embedding dictionary for category families
    embedding_dict = {}
    for family in df['category_family'].unique():
        if family in model.wv:
            embedding_dict[family] = model.wv[family]
    
    return {
        'embeddings': embedding_dict,
        'model': model,
        'dimension': vector_size
    }

fasttext_results = create_fasttext_embeddings(df)
print(f"FastText embeddings created with dimension {fasttext_results['dimension']}")
print(f"Number of families with embeddings: {len(fasttext_results['embeddings'])}")
print(f"Sample embeddings:")
for family, embedding in list(fasttext_results['embeddings'].items())[:3]:
    print(f"{family}: {embedding[:5]}... (showing first 5 of {len(embedding)} dimensions)")

## Embedding Comparison and Evaluation

In [ ]:
def compare_embeddings(embedding_results: List[Dict]) -> pd.DataFrame:
    """
    Compare different embedding techniques.
    """
    comparison_data = []
    
    for i, result in enumerate(embedding_results):
        embedding_dict = result['embeddings']
        dimension = result['dimension']
        
        # Calculate basic statistics
        all_embeddings = np.array(list(embedding_dict.values()))
        
        comparison_data.append({
            'method': ['Label Encoding', 'One-Hot', 'Feature-Based', 'Word2Vec', 'FastText'][i],
            'dimension': dimension,
            'num_families': len(embedding_dict),
            'mean_norm': np.mean(np.linalg.norm(all_embeddings, axis=1)),
            'std_norm': np.std(np.linalg.norm(all_embeddings, axis=1))
        })
    
    return pd.DataFrame(comparison_data)

# Collect all embedding results
all_results = [label_results, onehot_results, feature_results, word2vec_results, fasttext_results]

# Compare embeddings
comparison_df = compare_embeddings(all_results)
print("Embedding Comparison:")
comparison_df

## Visualization of Embeddings

In [ ]:
def visualize_embeddings(embedding_results: Dict, 
                         method_name: str,
                         technique: str = 'tsne') -> None:
    """
    Visualize embeddings using t-SNE or PCA.
    """
    embedding_dict = embedding_results['embeddings']
    
    # Prepare data
    families = list(embedding_dict.keys())
    embeddings = np.array(list(embedding_dict.values()))
    
    # Skip if dimension is 1 (label encoding)
    if embeddings.shape[1] == 1:
        print(f"Skipping {method_name} - 1D embeddings cannot be visualized with {technique.upper()}")
        return
    
    # Apply dimensionality reduction
    if technique == 'tsne':
        reducer = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=min(30, len(families)-1))
    else:  # PCA
        reducer = PCA(n_components=2, random_state=RANDOM_SEED)
    
    embeddings_2d = reducer.fit_transform(embeddings)
    
    # Create plot
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                         c=range(len(families)), cmap='viridis', s=100, alpha=0.7)
    
    # Add labels
    for i, family in enumerate(families):
        plt.annotate(family, (embeddings_2d[i, 0], embeddings_2d[i, 1]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=10)
    
    plt.title(f'{method_name} - {technique.upper()} Visualization')
    plt.xlabel(f'{technique.upper()} Component 1')
    plt.ylabel(f'{technique.upper()} Component 2')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Visualize different embedding methods
for result, name in zip(all_results, ['Label Encoding', 'One-Hot', 'Feature-Based', 'Word2Vec', 'FastText']):
    if result['dimension'] > 1:
        visualize_embeddings(result, name, 'tsne')
        visualize_embeddings(result, name, 'pca')

## Similarity Analysis

In [ ]:
def find_similar_families(target_family: str, 
                         embedding_results: Dict,
                         top_k: int = 3) -> List[Tuple[str, float]]:
    """
    Find similar category families using cosine similarity.
    """
    embedding_dict = embedding_results['embeddings']
    
    if target_family not in embedding_dict:
        return []
    
    target_embedding = embedding_dict[target_family].reshape(1, -1)
    
    similarities = []
    for family, embedding in embedding_dict.items():
        if family != target_family:
            similarity = cosine_similarity(target_embedding, embedding.reshape(1, -1))[0][0]
            similarities.append((family, similarity))
    
    # Sort by similarity and return top-k
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

# Test similarity with Word2Vec embeddings
print("Similarity Analysis using Word2Vec embeddings:")
print("=" * 50)

test_families = ['electronics', 'furniture', 'clothing']
for family in test_families:
    if family in word2vec_results['embeddings']:
        similar = find_similar_families(family, word2vec_results, top_k=3)
        print(f"\nFamilies similar to '{family}':")
        for similar_family, similarity in similar:
            print(f"  {similar_family}: {similarity:.3f}")

## Save Embeddings

In [ ]:
import pickle
import json
from pathlib import Path

def save_embeddings(embedding_results: Dict, 
                    method_name: str,
                    save_dir: str = './embeddings') -> None:
    """
    Save embeddings to disk.
    """
    save_path = Path(save_dir)
    save_path.mkdir(exist_ok=True)
    
    # Save embeddings as numpy array
    embedding_dict = embedding_results['embeddings']
    families = list(embedding_dict.keys())
    embeddings = np.array(list(embedding_dict.values()))
    
    # Save embeddings and metadata
    np.save(save_path / f'{method_name.lower().replace(" ", "_")}_embeddings.npy', embeddings)
    np.save(save_path / f'{method_name.lower().replace(" ", "_")}_families.npy', np.array(families))
    
    # Save metadata
    metadata = {
        'method': method_name,
        'dimension': embedding_results['dimension'],
        'num_families': len(families),
        'families': families
    }
    
    with open(save_path / f'{method_name.lower().replace(" ", "_")}_metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # Save the full results object (including encoders, models, etc.)
    with open(save_path / f'{method_name.lower().replace(" ", "_")}_full_results.pkl', 'wb') as f:
        pickle.dump(embedding_results, f)
    
    print(f"Saved {method_name} embeddings to {save_path}")

# Save all embedding methods
for result, name in zip(all_results, ['Label Encoding', 'One-Hot', 'Feature-Based', 'Word2Vec', 'FastText']):
    save_embeddings(result, name)

## Load and Use Embeddings

In [ ]:
def load_embeddings(method_name: str, 
                   load_dir: str = './embeddings') -> Dict:
    """
    Load embeddings from disk.
    """
    load_path = Path(load_dir)
    filename = method_name.lower().replace(' ', '_')
    
    # Load full results
    with open(load_path / f'{filename}_full_results.pkl', 'rb') as f:
        results = pickle.load(f)
    
    return results

# Example: Load and use Word2Vec embeddings
print("Loading Word2Vec embeddings...")
loaded_word2vec = load_embeddings('Word2Vec')

# Test loading
print(f"Loaded {len(loaded_word2vec['embeddings'])} embeddings")
print(f"Dimension: {loaded_word2vec['dimension']}")

# Example usage
if 'electronics' in loaded_word2vec['embeddings']:
    electronics_embedding = loaded_word2vec['embeddings']['electronics']
    print(f"\nElectronics embedding shape: {electronics_embedding.shape}")
    print(f"Electronics embedding (first 5 dims): {electronics_embedding[:5]}")

## Summary and Recommendations

### Embedding Methods Comparison:

1. **Label Encoding**:
   - Pros: Simple, memory efficient
   - Cons: Ordinal relationship assumption, limited expressiveness
   - Best for: Tree-based models, simple categorical features

2. **One-Hot Encoding**:
   - Pros: No ordinal assumptions, interpretable
   - Cons: High dimensionality, sparse
   - Best for: Linear models, small number of categories

3. **Feature-Based Embeddings**:
   - Pros: Incorporates domain knowledge, meaningful dimensions
   - Cons: Depends on feature quality, requires feature engineering
   - Best for: When you have rich product features

4. **Word2Vec Embeddings**:
   - Pros: Captures semantic relationships, dense representations
   - Cons: Requires training, less interpretable
   - Best for: When you have hierarchical category data

5. **FastText Embeddings**:
   - Pros: Handles new categories, robust to misspellings
   - Cons: More complex training
   - Best for: Dynamic category systems, noisy data

### Recommendations:

- **For production systems**: Start with feature-based embeddings, experiment with Word2Vec
- **For interpretability**: Use feature-based or one-hot encodings
- **For deep learning**: Use Word2Vec or FastText embeddings
- **For small datasets**: One-hot encoding may be sufficient
- **For large-scale systems**: Consider pre-trained embeddings or custom training

In [ ]:
# Final summary
print("\n" + "="*60)
print("CATEGORY FAMILY EMBEDDINGS - SUMMARY")
print("="*60)

print(f"\nDataset processed: {df.shape[0]} products")
print(f"Unique category families: {df['category_family'].nunique()}")
print(f"\nEmbedding methods created: {len(all_results)}")

for i, (result, name) in enumerate(zip(all_results, ['Label Encoding', 'One-Hot', 'Feature-Based', 'Word2Vec', 'FastText'])):
    print(f"{i+1}. {name}: {result['dimension']}D, {len(result['embeddings'])} families")

print(f"\nEmbeddings saved to: ./embeddings/")
print("\nNext steps:")
print("1. Choose appropriate embedding method for your use case")
print("2. Integrate embeddings into your ML pipeline")
print("3. Monitor embedding performance over time")
print("4. Consider retraining embeddings as categories evolve")

print("\n" + "="*60)